In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("../dataset/train.csv")

In [3]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
import re

def clean_text(text):

    text = text.lower()

    # Replace URLs with token
    text = re.sub(r'http\S+|www\S+', ' URL ', text)

    # Remove @ but keep username
    text = re.sub(r'@(\w+)', r'\1', text)

    # Remove # but keep hashtag word
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove punctuation except underscores
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [5]:
df["clean_text"] = df["text"].apply(clean_text)

In [6]:
df[["text", "clean_text"]].head(10)

,text,clean_text
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...
3,"13,000 people receive #wildfires evacuation or...",13 000 people receive wildfires evacuation ord...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...
5,#RockyFire Update => California Hwy. 20 closed...,rockyfire update california hwy 20 closed in b...
6,#flood #disaster Heavy rain causes flash flood...,flood disaster heavy rain causes flash floodin...
7,I'm on top of the hill and I can see a fire in...,i m on top of the hill and i can see a fire in...
8,There's an emergency evacuation happening now ...,there s an emergency evacuation happening now ...
9,I'm afraid that the tornado is coming to our a...,i m afraid that the tornado is coming to our area


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["clean_text"])

mask = (df["target"] == 1).to_numpy()

word_freq = X[mask].sum(axis=0)

words = vectorizer.get_feature_names_out()

freq_df = pd.DataFrame({
    "word": words,
    "count": word_freq.A1
})

freq_df.sort_values("count", ascending=False).head(20)

,word,count
15632,url,2521
16696,û_,172
10269,news,138
1113,amp,135
4522,disaster,121
2683,california,115
14296,suicide,112
11432,police,109
11138,people,105
8367,killed,95


In [8]:
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2,2)
)

X = vectorizer.fit_transform(
    df[df["target"]==1]["clean_text"]
)

bigrams = pd.DataFrame({
    "bigram": vectorizer.get_feature_names_out(),
    "count": X.sum(axis=0).A1
})

bigrams.sort_values(
    "count",
    ascending=False
).head(20)

,bigram,count
19039,url url,248
20560,û_ url,121
17148,suicide bomber,59
12732,northern california,41
12993,oil spill,38
3098,burning buildings,37
17150,suicide bombing,35
3288,california wildfire,34
706,70 years,30
2688,bomber detonated,30


In [9]:
import nltk

nltk.download("punkt")#used for splitting text into words or sent
nltk.download("averaged_perceptron_tagger_eng")#pos model
nltk.download("wordnet")#wordnet dict for lemma
nltk.download("omw-1.4")#open multilingual wordnet , multiple lang for lemma

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [10]:
from nltk.corpus import wordnet

def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    return wordnet.NOUN

In [11]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    tagged_words = pos_tag(words)

    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_words
    ]

    return " ".join(lemmas)

In [12]:
df["lemma_text"] = df["clean_text"].apply(lemmatize_text)

In [13]:
df[["clean_text", "lemma_text"]].head(10)

#people are trapped -> people be trap
#building destroyed -> building destroy
#hence used lemma

,clean_text,lemma_text
0,our deeds are the reason of this earthquake ma...,our deed be the reason of this earthquake may ...
1,forest fire near la ronge sask canada,forest fire near la ronge sask canada
2,all residents asked to shelter in place are be...,all resident ask to shelter in place be be not...
3,13 000 people receive wildfires evacuation ord...,13 000 people receive wildfire evacuation orde...
4,just got sent this photo from ruby alaska as s...,just get send this photo from ruby alaska a sm...
5,rockyfire update california hwy 20 closed in b...,rockyfire update california hwy 20 close in bo...
6,flood disaster heavy rain causes flash floodin...,flood disaster heavy rain cause flash flooding...
7,i m on top of the hill and i can see a fire in...,i m on top of the hill and i can see a fire in...
8,there s an emergency evacuation happening now ...,there s an emergency evacuation happen now in ...
9,i m afraid that the tornado is coming to our area,i m afraid that the tornado be come to our area


In [14]:
(df["clean_text"] != df["lemma_text"]).sum()

np.int64(6481)

In [15]:
vocab = set()

for text in df["clean_text"]:
    vocab.update(text.split())

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 17081


In [16]:
vocab = set()

for text in df["lemma_text"]:
    vocab.update(text.split())

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 15040


In [17]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["target"] # train test sets maintain same ratio of target that is disaster or non disaster
)

y_train = df.loc[train_idx, "target"]
y_test = df.loc[test_idx, "target"]

X_train_clean = df.loc[train_idx, "clean_text"]
X_test_clean = df.loc[test_idx, "clean_text"]

X_train_lemma = df.loc[train_idx, "lemma_text"]
X_test_lemma = df.loc[test_idx, "lemma_text"]

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,3))),
    ("lr", LogisticRegression(max_iter=1000)),
    #("nb", MultinomialNB())
    #("svm", LinearSVC())

])


param_grid = {

    "tfidf__max_features": [5000, 10000, 15000],

    "tfidf__min_df": [2, 3],

    "tfidf__max_df": [0.85, 0.90, 0.95],

    "tfidf__sublinear_tf": [True, False],

    "lr__C": [0.01, 0.1, 1, 10, 100]
    #"svm__C"
}



In [19]:
from sklearn.model_selection import GridSearchCV

grid_clean = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid_clean.fit(X_train_clean, y_train)

print(grid_clean.best_params_)
print(grid_clean.best_score_)

Fitting 5 folds for each of 180 candidates, totalling 900 fits
{'lr__C': 1, 'tfidf__max_df': 0.85, 'tfidf__max_features': 10000, 'tfidf__min_df': 3, 'tfidf__sublinear_tf': True}
0.7397286355689581


In [20]:
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

pred_clean = grid_clean.predict(X_test_clean)

print(classification_report(y_test, pred_clean))



              precision    recall  f1-score   support

           0       0.80      0.89      0.84       869
           1       0.83      0.71      0.77       654

    accuracy                           0.81      1523
   macro avg       0.82      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523



In [21]:
grid_lemma = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid_lemma.fit(X_train_lemma, y_train)

print(grid_lemma.best_params_)
print(grid_lemma.best_score_)

Fitting 5 folds for each of 180 candidates, totalling 900 fits
{'lr__C': 1, 'tfidf__max_df': 0.85, 'tfidf__max_features': 5000, 'tfidf__min_df': 3, 'tfidf__sublinear_tf': True}
0.7429223643535421


In [22]:
pred_lemma = grid_lemma.predict(X_test_lemma)

print(classification_report(y_test, pred_lemma))



              precision    recall  f1-score   support

           0       0.82      0.89      0.85       869
           1       0.83      0.73      0.78       654

    accuracy                           0.82      1523
   macro avg       0.82      0.81      0.81      1523
weighted avg       0.82      0.82      0.82      1523



In [23]:
df.to_csv("../dataset/disaster_tweets_processed.csv", index=False)

In [24]:
import joblib

joblib.dump(
    grid_lemma.best_estimator_,
    "../model/disaster_pipeline.pkl"
)

['../model/disaster_pipeline.pkl']

In [25]:
df_train = pd.read_csv("../dataset/train.csv")

df_train["clean_text"] = df_train["text"].apply(clean_text)
df_train["lemma_text"] = df_train["clean_text"].apply(lemmatize_text)

In [26]:
df_test = pd.read_csv("../dataset/test.csv")

df_test["clean_text"] = df_test["text"].apply(clean_text)
df_test["lemma_text"] = df_test["clean_text"].apply(lemmatize_text)

In [27]:
X_train_final = df_train["lemma_text"]
y_final = df_train["target"]

grid_lemma.fit(X_train_final, y_final)

Fitting 5 folds for each of 180 candidates, totalling 900 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'lr__C': [0.01, 0.1, ...], 'tfidf__max_df': [0.85, 0.9, ...], 'tfidf__max_features': [5000, 10000, ...], 'tfidf__min_df': [2, 3], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr

In [28]:
X_test_final = df_test["lemma_text"]

pred_final = grid_lemma.predict(X_test_final)

In [29]:
submission = pd.DataFrame({
    "id": df_test["id"],
    "target": pred_final
})

submission.head()

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


In [30]:
submission.to_csv(
    "../dataset/submission.csv",
    index=False
)